# Image processing in the browser

This notebook performs image processing — edge detection, filtering,
segmentation — entirely in your browser using only NumPy, SciPy, and
Matplotlib. No additional packages need to be installed.

SciPy's `ndimage` module provides the image processing primitives, all
compiled to WebAssembly via [emscripten-forge](https://emscripten-forge.org/).

## Generate a test image

Create a synthetic image with geometric shapes using pure NumPy — no external
files needed, since MEMFS doesn't persist across reloads.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

rng = np.random.default_rng(42)

# Create a 300x300 grayscale image with geometric shapes
img = np.zeros((300, 300), dtype=np.float64)

# Circle
yy, xx = np.ogrid[:300, :300]
circle = ((xx - 80)**2 + (yy - 80)**2) < 50**2
img[circle] = 0.9

# Rectangle
img[60:180, 180:280] = 0.7

# Triangle (using a mask)
for row in range(120, 250):
    half_width = int((row - 120) * 0.6)
    center = 150
    img[row, max(0, center - half_width):min(300, center + half_width)] = 0.5

# Small circle
small_circle = ((xx - 250)**2 + (yy - 250)**2) < 30**2
img[small_circle] = 0.8

# Add slight noise
img += rng.normal(scale=0.03, size=img.shape)
img = np.clip(img, 0, 1)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(img, cmap="gray", vmin=0, vmax=1)
ax.set_title("Synthetic test image (NumPy)")
ax.axis("off")
plt.tight_layout()
plt.show()
print(f"Image shape: {img.shape}, dtype: {img.dtype}")

In [ ]:
## Edge detection

Compute image gradients with Sobel operators from `scipy.ndimage`, then
combine into an edge magnitude map.

In [ ]:
sx = ndimage.sobel(img, axis=1)
sy = ndimage.sobel(img, axis=0)
edges = np.hypot(sx, sy)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(sx, cmap="RdBu_r", vmin=-0.5, vmax=0.5)
axes[0].set_title("Sobel X (horizontal edges)")
axes[1].imshow(sy, cmap="RdBu_r", vmin=-0.5, vmax=0.5)
axes[1].set_title("Sobel Y (vertical edges)")
axes[2].imshow(edges, cmap="gray")
axes[2].set_title("Edge magnitude")

for ax in axes:
    ax.axis("off")
fig.suptitle("Edge detection — scipy.ndimage in WebAssembly", fontsize=13)
plt.tight_layout()
plt.show()

## Spatial filtering

Apply Gaussian smoothing and sharpening via convolution — fundamental
image processing operations from `scipy.ndimage`.

In [ ]:
blurred_2 = ndimage.gaussian_filter(img, sigma=2)
blurred_5 = ndimage.gaussian_filter(img, sigma=5)
sharpened = img + 2.0 * (img - blurred_2)
sharpened = np.clip(sharpened, 0, 1)

filter_results = [
    ("Original", img),
    ("Gaussian σ=2", blurred_2),
    ("Gaussian σ=5", blurred_5),
    ("Sharpened (unsharp mask)", sharpened),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, result) in zip(axes, filter_results):
    ax.imshow(result, cmap="gray", vmin=0, vmax=1)
    ax.set_title(name)
    ax.axis("off")

fig.suptitle("Spatial filters — scipy.ndimage in WebAssembly", fontsize=13)
plt.tight_layout()
plt.show()

## Morphological operations

Binary morphology — erosion, dilation, opening — using `scipy.ndimage`.
These are the building blocks for shape analysis and noise removal.

In [ ]:
# Threshold to binary
binary = img > 0.3

struct = ndimage.generate_binary_structure(2, 1)
eroded = ndimage.binary_erosion(binary, structure=struct, iterations=3)
dilated = ndimage.binary_dilation(binary, structure=struct, iterations=3)
opened = ndimage.binary_opening(binary, structure=struct, iterations=3)

morph_results = [
    ("Binary (threshold > 0.3)", binary),
    ("Eroded (3 iterations)", eroded),
    ("Dilated (3 iterations)", dilated),
    ("Opened (erosion + dilation)", opened),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, result) in zip(axes, morph_results):
    ax.imshow(result, cmap="gray")
    ax.set_title(name, fontsize=10)
    ax.axis("off")

fig.suptitle("Binary morphology — scipy.ndimage in WebAssembly", fontsize=13)
plt.tight_layout()
plt.show()

## Segmentation

Threshold the image and use connected-component labeling from
`scipy.ndimage` to identify distinct regions.

In [ ]:
labels, n_features = ndimage.label(opened)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))

ax1.imshow(img, cmap="gray", vmin=0, vmax=1)
ax1.set_title("Original")

ax2.imshow(opened, cmap="gray")
ax2.set_title("After morphological opening")

ax3.imshow(labels, cmap="nipy_spectral")
ax3.set_title(f"Connected components ({n_features} regions)")

# Draw bounding boxes around detected regions
for i in range(1, n_features + 1):
    region = ndimage.find_objects(labels)[i - 1]
    y_slice, x_slice = region
    rect = plt.Rectangle(
        (x_slice.start, y_slice.start),
        x_slice.stop - x_slice.start,
        y_slice.stop - y_slice.start,
        fill=False, edgecolor="white", linewidth=1.5,
    )
    ax3.add_patch(rect)

for ax in (ax1, ax2, ax3):
    ax.axis("off")

fig.suptitle("Image segmentation — scipy.ndimage in WebAssembly", fontsize=13)
plt.tight_layout()
plt.show()

# Per-region statistics
print(f"Found {n_features} connected regions:")
for i in range(1, n_features + 1):
    area = np.sum(labels == i)
    cy, cx = ndimage.center_of_mass(labels == i)
    print(f"  Region {i}: area={area:>5d} px, centroid=({cx:.0f}, {cy:.0f})")

## What just happened

You performed edge detection, spatial filtering, morphological operations,
and connected-component segmentation — all using SciPy and NumPy running as
WebAssembly in your browser tab.

For more advanced image processing, you can install
[Pillow](https://pillow.readthedocs.io/) or
[scikit-image](https://scikit-image.org/) from emscripten-forge using
`%conda install pillow` — see the
[c-extensions notebook](c-extensions.ipynb) for how runtime installation works.